# Noisy Simple-Crystal Dataset Inspection

This notebook inspects the generated noisy FCC/HCP/BCC/SC/diamond benchmark and then checks how well q_l features separate the known crystal labels.

In [ ]:

from pathlib import Path
import json
import math
import random
import sys

import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({"figure.figsize": (7, 5), "axes.grid": True})


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "lammps").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not find autoencode_statmech repo root from current working directory")


REPO_ROOT = find_repo_root()
OTHER_DATA = REPO_ROOT / "lammps/LJ/statistically_independent_samples/statistically_independent_samples/other_data"
print(f"Repo root: {REPO_ROOT}")
print(f"Other data: {OTHER_DATA}")

try:
    import freud
except ImportError as exc:
    raise ImportError("This notebook needs freud: conda install -c conda-forge freud or python -m pip install freud-analysis") from exc

DATA_DIR = OTHER_DATA / "noisy_simple_crystals"
NPZ_PATH = DATA_DIR / "noisy_crystals.npz"
META_PATH = DATA_DIR / "metadata.json"
if not NPZ_PATH.exists():
    raise FileNotFoundError(f"Missing generated noisy crystal data: {NPZ_PATH}. Run download_q_ls_hard_data.py generate-noisy-crystals first.")

data = np.load(NPZ_PATH)
metadata = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}
structures = [str(item) for item in data["structures"]]
noise_levels = np.asarray(data["noise_levels"], dtype=float)
print("structures:", structures)
print("noise levels:", noise_levels)
print("metadata parameters:", metadata.get("parameters", {}))


## Dataset Overview

In [ ]:

print(f"{'structure':12s} {'snapshots':>10s} {'particles':>10s} {'box example':>28s}")
for structure in structures:
    positions = data[f"positions_{structure}"]
    boxes = data[f"boxes_{structure}"]
    print(f"{structure:12s} {positions.shape[0]:10d} {positions.shape[1]:10d} {str(tuple(np.round(boxes[0], 3))):>28s}")


## Visualize Noisy Structures

Use `NOISE_INDEX = 0` for ideal structures and `NOISE_INDEX = -1` for the noisiest structures.

In [ ]:

RNG = np.random.default_rng(7)
NOISE_INDEX = -1
SAMPLE_INDEX_WITHIN_NOISE = 0
MAX_POINTS = 2500


def set_axes_equal(ax, positions):
    mins = positions.min(axis=0)
    maxs = positions.max(axis=0)
    centers = 0.5 * (mins + maxs)
    radius = 0.55 * float(np.max(maxs - mins))
    for dim, setter in enumerate((ax.set_xlim, ax.set_ylim, ax.set_zlim)):
        setter(centers[dim] - radius, centers[dim] + radius)


def snapshot_index_for_noise(structure, noise_index=-1, sample_index=0):
    noise = data[f"noise_{structure}"]
    levels = sorted(set(float(x) for x in noise))
    target_noise = levels[noise_index]
    candidates = np.where(np.isclose(noise, target_noise))[0]
    return int(candidates[min(sample_index, len(candidates) - 1)]), target_noise


def plot_noisy_structures(noise_index=-1, sample_index=0, max_points=2500):
    n = len(structures)
    cols = min(3, n)
    rows = math.ceil(n / cols)
    fig = plt.figure(figsize=(4.8 * cols, 4.2 * rows))
    for i, structure in enumerate(structures, start=1):
        snap_idx, noise = snapshot_index_for_noise(structure, noise_index, sample_index)
        positions = data[f"positions_{structure}"][snap_idx]
        if len(positions) > max_points:
            idx = RNG.choice(len(positions), size=max_points, replace=False)
            show = positions[idx]
        else:
            show = positions
        ax = fig.add_subplot(rows, cols, i, projection="3d")
        ax.scatter(show[:, 0], show[:, 1], show[:, 2], c=show[:, 2], s=4, cmap="viridis", alpha=0.75)
        ax.set_title(f"{structure}: noise={noise:.3f}, N={len(positions)}")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        set_axes_equal(ax, show)
    fig.tight_layout()
    return fig

plot_noisy_structures(NOISE_INDEX, SAMPLE_INDEX_WITHIN_NOISE, MAX_POINTS)


## Compute q_l Features

In [ ]:

L_LIST = [4, 6, 8, 10, 12]
NUM_NEIGHBORS = 12
AVERAGE_Q = False
MAX_SNAPSHOTS_PER_STRUCTURE = 18
MAX_PARTICLES_PER_SNAPSHOT = 450
RANDOM_SEED = 11

rng = np.random.default_rng(RANDOM_SEED)


def freud_box_from_lengths(lengths):
    Lx, Ly, Lz = np.asarray(lengths, dtype=float)[:3]
    return freud.box.Box(Lx=Lx, Ly=Ly, Lz=Lz)


def sampled_snapshot_indices(n_snapshots, max_snapshots):
    if n_snapshots <= max_snapshots:
        return list(range(n_snapshots))
    return sorted(set(np.linspace(0, n_snapshots - 1, max_snapshots, dtype=int).tolist()))


def build_q_dataset():
    order = freud.order.Steinhardt(l=L_LIST, average=AVERAGE_Q, wl=False)
    X, y, noise_meta, snapshot_meta, particle_meta = [], [], [], [], []
    for structure in structures:
        positions_all = data[f"positions_{structure}"]
        boxes_all = data[f"boxes_{structure}"]
        noise_all = data[f"noise_{structure}"]
        for snap_idx in sampled_snapshot_indices(len(positions_all), MAX_SNAPSHOTS_PER_STRUCTURE):
            box_lengths = np.asarray(boxes_all[snap_idx], dtype=np.float32)
            positions = np.asarray(positions_all[snap_idx], dtype=np.float32) - 0.5 * box_lengths[:3]
            box = freud_box_from_lengths(box_lengths)
            order.compute(system=(box, positions), neighbors={"num_neighbors": NUM_NEIGHBORS})
            q = np.asarray(order.particle_order, dtype=np.float32)
            if len(q) > MAX_PARTICLES_PER_SNAPSHOT:
                idx = rng.choice(len(q), size=MAX_PARTICLES_PER_SNAPSHOT, replace=False)
            else:
                idx = np.arange(len(q))
            X.append(q[idx])
            y.extend([structure] * len(idx))
            noise_meta.extend([float(noise_all[snap_idx])] * len(idx))
            snapshot_meta.extend([snap_idx] * len(idx))
            particle_meta.extend(idx.tolist())
    X = np.vstack(X)
    y = np.asarray(y)
    meta = {
        "noise": np.asarray(noise_meta),
        "snapshot": np.asarray(snapshot_meta),
        "particle": np.asarray(particle_meta),
    }
    return X, y, meta

X_q, y_q, meta_q = build_q_dataset()
print("X_q shape:", X_q.shape)
print("labels:", {label: int(np.sum(y_q == label)) for label in sorted(set(y_q))})
print("noise levels in q dataset:", sorted(set(np.round(meta_q["noise"], 4))))


## q_l Separation Plots

In [ ]:

q4_idx = L_LIST.index(4)
q6_idx = L_LIST.index(6)
fig, ax = plt.subplots(figsize=(7, 5))
for label in sorted(set(y_q)):
    mask = y_q == label
    take = np.where(mask)[0]
    if len(take) > 1000:
        take = rng.choice(take, size=1000, replace=False)
    ax.scatter(X_q[take, q4_idx], X_q[take, q6_idx], s=8, alpha=0.5, label=label)
ax.set_xlabel("q4")
ax.set_ylabel("q6")
ax.set_title("Per-particle q4/q6 by noisy-crystal structure")
ax.legend(markerscale=2, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()

X_scaled = StandardScaler().fit_transform(X_q)
X_pca = PCA(n_components=2, random_state=RANDOM_SEED).fit_transform(X_scaled)
fig, ax = plt.subplots(figsize=(7, 5))
for label in sorted(set(y_q)):
    mask = y_q == label
    take = np.where(mask)[0]
    if len(take) > 1000:
        take = rng.choice(take, size=1000, replace=False)
    ax.scatter(X_pca[take, 0], X_pca[take, 1], s=8, alpha=0.5, label=label)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("PCA of sampled q_l vectors")
ax.legend(markerscale=2, fontsize=9, bbox_to_anchor=(1.02, 1), loc="upper left")
fig.tight_layout()


## q_l Classifier Baseline

In [ ]:

X_train, X_test, y_train, y_test, noise_train, noise_test = train_test_split(
    X_q,
    y_q,
    meta_q["noise"],
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=y_q,
)

clf = RandomForestClassifier(
    n_estimators=250,
    min_samples_leaf=3,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred, digits=3))
labels = sorted(set(y_q))
cm = confusion_matrix(y_test, y_pred, labels=labels, normalize="true")
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=labels).plot(ax=ax, xticks_rotation=45, values_format=".2f", cmap="Blues")
ax.set_title("q_l random-forest baseline, normalized confusion matrix")
fig.tight_layout()

# Accuracy by noise level shows where q_l starts to degrade.
noise_levels_seen = sorted(set(np.round(noise_test, 6)))
acc_by_noise = []
for noise in noise_levels_seen:
    mask = np.isclose(noise_test, noise)
    acc_by_noise.append(float(np.mean(y_pred[mask] == y_test[mask])))
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(noise_levels_seen, acc_by_noise, marker="o")
ax.set_xlabel("noise sigma / lattice constant")
ax.set_ylabel("test accuracy")
ax.set_ylim(0, 1.03)
ax.set_title("q_l classifier accuracy vs noise")
fig.tight_layout()

importances = clf.feature_importances_
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar([f"q{l}" for l in L_LIST], importances)
ax.set_ylabel("feature importance")
ax.set_title("Random-forest q_l feature importance")
fig.tight_layout()
